# GalactICS → ntropy: NFW halo end-to-end

*A walkthrough notebook for the PyGalactICS rewrite — from graduate-school Fortran to a testable Python library.*

## What is GalactICS?

**GalactICS** (Kuijken & Dubinski) is a code package for building **N-body initial conditions (ICs)** for idealized galaxy models — not a time integrator. Its job is to produce particle positions and velocities that are as close as practical to **dynamical equilibrium** in a chosen gravitational potential, so downstream simulations (Gadget, GIZMO, etc.) start from a physically meaningful state rather than an arbitrary point cloud.

At a high level GalactICS:

1. **Defines a galaxy model** — halo (often NFW), stellar disk(s), bulge, gas, each with parametric density laws.
2. **Solves for the self-consistent potential** — the `dbh` program iterates a multipole Poisson solver until the density implied by distribution functions (DFs) and analytic components matches the potential they generate (tidal radius convergence).
3. **Builds distribution functions** — halos and bulges use **Eddington inversion** on spherical ρ(r); disks use action-based DFs with asymmetric-drift corrections (`diskdf`).
4. **Samples particles** — `genhalo`, `gendisk`, `genbulge` draw phase-space points from those DFs and write ASCII particle files.

GalactICS assumes **G = 1** with length in kpc, velocity in 100 km/s, and a fixed mass unit (2.325×10⁹ M☉). That is the same unit system used throughout this repository.

## What this notebook does

This notebook is the **integrated test** the Python rewrite was built for:

| Stage | Tool | Purpose |
|-------|------|---------|
| IC generation | **galacticsics** + legacy `dbh` / `genhalo` | Real GalactICS pipeline via a typed Python API |
| Validation | **ntropy** | Short self-gravitating evolution to check IC quality |

We walk through a **spherical NFW halo** (simplest non-trivial case), then:

1. Sample halo particles through `dbh` → `genhalo`
2. Plot the initial **ρ(r)** to confirm the sampled profile looks like a halo
3. Benchmark **brute-force** vs **Barnes–Hut** gravity (force accuracy vs θ)
4. Measure **parallelism scaling** — time per force step vs N and vs MPI rank count; compare **order-1** vs **order-2** leapfrog
5. Evolve the IC briefly, track **|ΔE/E₀|** (Binney & Tremaine energy-conservation diagnostic), and plot **ρ(r)** to see whether the profile is stable

Figures and particle files are written to `notebooks/artifacts/` (gitignored).

**Prerequisites:** `make install-dev` (builds `legacy/bin`, installs both packages), plus `pip install jupyter matplotlib`.

## Repository layout (after the rewrite)

The graduate-school Fortran/C still runs, but Python owns the API and tests:

```
GalaxyModel  →  dbh (solve_potential)  →  genhalo  →  ParticleSet  →  ntropy Simulation
     ↑              legacy/bin/              legacy/bin/              src/ntropy/
 galacticsics      subprocess bridge        subprocess bridge
```

- **`galacticsics`** (`src/galacticsics/`) — models, I/O, `solve_potential`, `sample_galaxy`, artifact generation.
- **`legacy/`** — original `dbh.f`, `genhalo`, etc. (gitignored; `make legacy-build`).
- **`ntropy`** (`src/ntropy/`) — minimal N-body integrator that reads the same ASCII particle format.

**Units** (shared everywhere):

| Quantity | Unit |
|----------|------|
| Length | kpc |
| Velocity | 100 km/s |
| Mass | 2.325×10⁹ M☉ |
| G | 1 |

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np

from ntropy.analysis.density import bin_spherical_density, compare_density_profiles
from ntropy.config import ForceConfig, IntegratorConfig, ParallelConfig, RunConfig
from ntropy.forces.brute import compute_forces_brute
from ntropy.forces.bhtree import compute_forces_bh
from ntropy.integrations.galacticsics import (
    galacticsics_available,
    nfw_halo_model_fast,
    sample_galacticsics_halo,
)
from ntropy.simulation import Simulation

if not galacticsics_available():
    raise RuntimeError(
        "galacticsics + legacy binaries required. From repo root: make install-dev"
    )

# Repo root (parent of notebooks/)
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
elif not (REPO / "src" / "ntropy").exists():
    REPO = REPO.parent  # running from notebooks/ subdir in some IDEs

ARTIFACTS = REPO / "notebooks" / "artifacts" / "nfw_walkthrough"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Keep matplotlib cache inside artifacts (useful in CI / sandboxed envs)
os.environ.setdefault("MPLCONFIGDIR", str(ARTIFACTS / ".matplotlib"))

import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

print(f"Artifacts → {ARTIFACTS}")

## 1. GalactICS: model → dbh → genhalo

### Pipeline steps

1. **`GalaxyModel`** — describes which components exist and the numerical grid (`nr`, `dr`, `lmax`).
2. **`solve_potential` (`dbh`)** — self-consistent multipole solve. For a pure halo with `lmax=0` this reduces to a spherical problem: build ρ(r) from the NFW law, invert the DF, tabulate `dfnfw.dat`, and write `dbh.dat` / `h.dat`.
3. **`genhalo`** — Monte Carlo sample of (r, v) from `dfnfw.dat`, output ASCII particles.

`sample_galacticsics_halo()` in `ntropy.integrations.galacticsics` wraps all three steps and converts the result to ntropy's `ParticleState`.

### `nfw_halo_model_fast()` — what it is and why

Notebooks and CI cannot wait for a full `nr=2000` production halo solve. The helper `nfw_halo_model_fast()` returns a **deliberately reduced** `GalaxyModel` that preserves the physics of an isolated NFW halo but finishes `dbh` in seconds:

| Parameter | Production-style (`nfw_halo_only`) | `nfw_halo_model_fast()` | Why reduced |
|-----------|-----------------------------------|-------------------------|-------------|
| `r_outer` (chalo) | 100 kpc | **60 kpc** | Smaller domain → fewer radial integrals |
| `a` (scale radius) | 10 kpc | **8 kpc** | Slightly more compact halo |
| `v0` | 2.0 (200 km/s) | **2.0** | Same potential depth scale |
| `dr_trunc` | 8–12 kpc | **8 kpc** | Outer truncation of NFW |
| `nr` (radial points) | 2000 | **800** | Coarser grid (like `reference_disk_halo`) |
| `lmax` | 0 | **0** | Purely spherical — no disk/bulge multipoles |

With `lmax=0` only the monopole (spherical) harmonic is active, which is correct for an isolated halo test. The model still runs the **real** `dbh` + Eddington DF machinery — it is not a toy analytic sampler.

**Softening** (`eps = 0.04` kpc below) is an ntropy parameter only; GalactICS particles are softened when we evolve them in ntropy, not during `genhalo`.

In [ ]:
model = nfw_halo_model_fast()
print(model)

GALACTICS_WORK = ARTIFACTS / "galacticsics_work"
EPS = 0.04

ic_result = sample_galacticsics_halo(
    model,
    n_particles=256,
    seed=-42,  # legacy ran3 convention (negative integer)
    work_dir=GALACTICS_WORK,
    eps=EPS,
    solve=True,
)

state = ic_result.state
particle_file = ARTIFACTS / "nfw_halo_galacticsics.dat"
state.write_ascii(particle_file)

print(f"N = {state.n}, total mass = {state.mass.sum():.2f}")
print(f"GalactICS work_dir = {ic_result.work_dir}")
print(f"dbh.dat present: {(ic_result.work_dir / 'dbh.dat').exists()}")
print(f"dfnfw.dat present: {(ic_result.work_dir / 'dfnfw.dat').exists()}")
print(f"Wrote {particle_file.name}")

## 2. Initial density plot — did `genhalo` sample the right halo?

### Why this plot?

GalactICS guarantees equilibrium only in the **smooth** DF limit. With finite N (here 256 particles) we expect Monte Carlo noise, but the **shape** of ρ(r) should still look like an NFW halo: roughly flat in the inner core, steepening toward the outer edge.

Binning particles into spherical shells and plotting ρ(r) answers:

- Did `dbh` + `genhalo` produce a spatial distribution consistent with an NFW profile?
- Are there obvious bugs in the Python bridge (wrong units, missing files, empty output)?

This is a **sanity check on the IC step**, before we spend time on dynamics.

### How we bin

Each shell receives the sum of particle masses between radii rᵢ and rᵢ₊₁; we divide by the shell volume 4π(rᵢ₊₁³ − rᵢ³)/3 to get a density estimate. We use log-log axes because NFW ρ(r) spans several decades.

In [ ]:
r_max = float(np.sqrt(np.sum(state.pos**2, axis=1)).max()) * 1.1
profile = bin_spherical_density(state.pos, state.mass, n_bins=20, r_max=r_max)

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(profile.r_mid, profile.rho, "o-", ms=5, alpha=0.8, label="genhalo particles")
ax.set_xlabel("r [kpc]")
ax.set_ylabel("ρ [M_unit / kpc³]")
ax.set_title("GalactICS NFW halo — initial ρ(r)")
ax.legend()
fig.tight_layout()
density_init_path = ARTIFACTS / "density_initial.png"
fig.savefig(density_init_path, dpi=150)
plt.show()
print(f"Saved {density_init_path.name}")

## 3. Force accuracy plots — can we trust the Barnes–Hut tree?

ntropy offers two force kernels. Before running a long simulation with the tree, we need to know **how wrong** it is compared to the exact pairwise sum.

| Method | Cost | Role in this notebook |
|--------|------|------------------------|
| **Brute force** | O(N²) | Reference truth at N = 256 |
| **Barnes–Hut** | O(N log N) | Production choice for larger N |

The tree opens a node when `size / distance ≥ θ` (larger θ → more aggressive grouping → faster but less accurate).

### Why these plots?

1. **Error vs θ (left panel)** — picks an operating point. We want the smallest θ that still gives acceptable error on this halo geometry. This is standard practice when calibrating tree codes for a new softening / particle layout.

2. **Error vs radius at fixed θ (right panel)** — reveals *where* the tree struggles. Inner regions with many close neighbours often show larger relative errors; the outskirts may be excellent. If errors were uniform in radius we might suspect a bug; radial structure is expected.

We report max, median, and RMS relative acceleration error ‖Δ**a**‖ / ‖**a**_brute‖ and save `force_accuracy.csv` for the blog post.

In [ ]:
pos, mass, eps = state.pos, state.mass, state.eps
acc_brute = compute_forces_brute(pos, mass, eps)
brute_norm = np.linalg.norm(acc_brute, axis=1)
brute_norm = np.maximum(brute_norm, 1e-30)

thetas = [0.2, 0.3, 0.4, 0.5, 0.7, 1.0]
rows = []

for theta in thetas:
    acc_bh = compute_forces_bh(pos, mass, eps, theta=theta)
    delta = acc_bh - acc_brute
    rel = np.linalg.norm(delta, axis=1) / brute_norm
    rows.append({
        "theta": theta,
        "max_rel": float(rel.max()),
        "median_rel": float(np.median(rel)),
        "rms_rel": float(np.sqrt(np.mean(rel**2))),
    })

print(f"{'theta':>6} {'max_rel':>10} {'median_rel':>12} {'rms_rel':>10}")
for row in rows:
    print(f"{row['theta']:6.1f} {row['max_rel']:10.4f} {row['median_rel']:12.4f} {row['rms_rel']:10.4f}")

csv_path = ARTIFACTS / "force_accuracy.csv"
with open(csv_path, "w") as f:
    f.write("theta,max_rel,median_rel,rms_rel\n")
    for row in rows:
        f.write(f"{row['theta']},{row['max_rel']},{row['median_rel']},{row['rms_rel']}\n")
print(f"\nSaved {csv_path.name}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

theta_arr = [r["theta"] for r in rows]
axes[0].semilogy(theta_arr, [r["max_rel"] for r in rows], "o-", label="max")
axes[0].semilogy(theta_arr, [r["median_rel"] for r in rows], "s-", label="median")
axes[0].set_xlabel("opening angle θ")
axes[0].set_ylabel("relative |Δa| / |a_brute|")
axes[0].set_title("BH force error vs θ")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Per-particle error at θ = 0.5
theta_demo = 0.5
acc_demo = compute_forces_bh(pos, mass, eps, theta=theta_demo)
rel_demo = np.linalg.norm(acc_demo - acc_brute, axis=1) / brute_norm
r = np.sqrt(np.sum(pos**2, axis=1))
axes[1].scatter(r, rel_demo, s=12, alpha=0.6, c=r, cmap="viridis")
axes[1].set_xlabel("r [kpc]")
axes[1].set_ylabel("relative force error")
axes[1].set_title(f"Per-particle error at θ = {theta_demo}")
fig.colorbar(axes[1].collections[0], ax=axes[1], label="r [kpc]")
fig.tight_layout()
force_plot_path = ARTIFACTS / "force_accuracy.png"
fig.savefig(force_plot_path, dpi=150)
plt.show()
print(f"Saved {force_plot_path.name}")

## 4. Parallelism and integrator performance

Gravity dominates cost in most N-body runs. This section times **one force evaluation** and **one integration step** on the GalactICS halo IC (resampled to larger N where needed).

### What we measure

| Benchmark | Varies | Reports |
|-----------|--------|---------|
| **Serial scaling** | Particle count N | ms per force step for **brute** (O(N²)) vs **Barnes–Hut** (O(N log N)) |
| **MPI scaling** | `mpirun -n` ranks (when available) | ms per force step at fixed N |
| **Integrator order** | Leapfrog order 1 vs 2 | ms per step and ms per force eval (order 2 calls the force kernel twice) |

**Integrators**

- **Order 1** — symplectic Euler: one kick using **a**(**x**ₙ), then drift with the updated velocity. One force evaluation per step; first-order accurate.
- **Order 2** — velocity Verlet (standard leapfrog): half-kick → drift → half-kick. Two force evaluations per step; second-order accurate.

**MPI note:** multi-rank benchmarks launch `notebooks/mpi_force_bench.py` via `mpirun`. If `mpi4py` or `mpirun` is missing, MPI panels are skipped and the notebook still runs the serial scaling and integrator comparison.

In [ ]:
import json
import shutil
import subprocess
import sys
import time

from ntropy.parallel.mpi import compute_forces_mpi, mpi_available
from ntropy.particles import ParticleState

BENCH_THETA = 0.5
BENCH_N_WARMUP = 4
BENCH_N_REPEAT = 25
MPI_BENCH_SCRIPT = REPO / "notebooks" / "mpi_force_bench.py"
BENCH_WORK = ARTIFACTS / "benchmarks"
BENCH_WORK.mkdir(parents=True, exist_ok=True)


def resample_state(base: ParticleState, n_target: int, rng: np.random.Generator) -> ParticleState:
    """Bootstrap-resample particles to a target count (with replacement if needed)."""
    replace = n_target > base.n
    idx = rng.choice(base.n, size=n_target, replace=replace)
    return ParticleState(
        pos=base.pos[idx],
        vel=base.vel[idx],
        mass=base.mass[idx],
        eps=base.eps[idx],
        tags=None if base.tags is None else base.tags[idx],
    )


def time_serial_force(
    pos: np.ndarray,
    mass: np.ndarray,
    eps: np.ndarray,
    method: str,
    *,
    theta: float = BENCH_THETA,
    n_warmup: int = BENCH_N_WARMUP,
    n_repeat: int = BENCH_N_REPEAT,
) -> float:
    """Wall time [s] for one serial force evaluation."""
    for _ in range(n_warmup):
        if method == "brute":
            compute_forces_brute(pos, mass, eps)
        else:
            compute_forces_bh(pos, mass, eps, theta=theta)
    start = time.perf_counter()
    for _ in range(n_repeat):
        if method == "brute":
            compute_forces_brute(pos, mass, eps)
        else:
            compute_forces_bh(pos, mass, eps, theta=theta)
    return (time.perf_counter() - start) / n_repeat


def time_mpi_dispatch_force(
    pos: np.ndarray,
    mass: np.ndarray,
    eps: np.ndarray,
    method: str,
    *,
    theta: float = BENCH_THETA,
    n_warmup: int = BENCH_N_WARMUP,
    n_repeat: int = BENCH_N_REPEAT,
) -> float:
    """Wall time [s] for one force evaluation via the MPI dispatch path (single rank)."""
    for _ in range(n_warmup):
        compute_forces_mpi(pos, mass, eps, method=method, theta=theta)
    start = time.perf_counter()
    for _ in range(n_repeat):
        compute_forces_mpi(pos, mass, eps, method=method, theta=theta)
    return (time.perf_counter() - start) / n_repeat


def mpi_scaling_available() -> bool:
    return mpi_available() and shutil.which("mpirun") is not None and MPI_BENCH_SCRIPT.exists()


def bench_mpi_ranks(
    n_ranks: int,
    pos: np.ndarray,
    mass: np.ndarray,
    eps: np.ndarray,
    method: str,
    *,
    theta: float = BENCH_THETA,
    n_repeat: int = 20,
) -> float:
    """Launch mpirun worker; return seconds per force evaluation."""
    tag = f"n{len(mass)}_{method}_r{n_ranks}"
    state_path = BENCH_WORK / f"{tag}.npz"
    out_path = BENCH_WORK / f"{tag}.json"
    np.savez(state_path, pos=pos, mass=mass, eps=eps)
    cmd = [
        "mpirun",
        "-n",
        str(n_ranks),
        sys.executable,
        str(MPI_BENCH_SCRIPT),
        str(state_path),
        method,
        str(theta),
        str(n_repeat),
        str(out_path),
    ]
    subprocess.run(cmd, check=True, cwd=REPO)
    payload = json.loads(out_path.read_text())
    return float(payload["time_per_force_s"])


def time_integration_step(
    base: ParticleState,
    *,
    method: str,
    order: int,
    parallel: bool = False,
    n_warmup: int = 3,
    n_repeat: int = 15,
) -> float:
    """Wall time [s] for one Simulation.step() including all force evaluations."""
    cfg = RunConfig()
    cfg.integrator = IntegratorConfig(dt=0.002, n_steps=1, order=order)
    cfg.force = ForceConfig(method=method, theta=BENCH_THETA)
    cfg.parallel = ParallelConfig(enabled=parallel)
    cfg.output.write_final = False
    cfg.output.every = 0

    for _ in range(n_warmup):
        sim = Simulation(cfg, state=base.copy())
        sim.step()
    start = time.perf_counter()
    for _ in range(n_repeat):
        sim = Simulation(cfg, state=base.copy())
        sim.step()
    return (time.perf_counter() - start) / n_repeat


# --- Serial scaling vs N ---
N_VALUES = [64, 128, 256, 512, 1024]
n_scaling_rows = []
for n_target in N_VALUES:
    sub = resample_state(state, n_target, rng)
    p, m, e = sub.pos, sub.mass, sub.eps
    t_brute = time_serial_force(p, m, e, "brute")
    t_bh = time_serial_force(p, m, e, "bh")
    n_scaling_rows.append({
        "n": n_target,
        "brute_ms": t_brute * 1e3,
        "bh_ms": t_bh * 1e3,
        "speedup_bh_over_brute": t_brute / t_bh,
    })

n_csv = ARTIFACTS / "force_scaling_vs_n.csv"
with open(n_csv, "w") as f:
    f.write("n,brute_ms,bh_ms,speedup_bh_over_brute\n")
    for row in n_scaling_rows:
        f.write(
            f"{row['n']},{row['brute_ms']:.6f},{row['bh_ms']:.6f},{row['speedup_bh_over_brute']:.4f}\n"
        )

print("Serial time per force step [ms]:")
print(f"{'N':>6} {'brute':>10} {'BH':>10} {'BH/brute':>10}")
for row in n_scaling_rows:
    print(
        f"{row['n']:6d} {row['brute_ms']:10.3f} {row['bh_ms']:10.3f} {row['speedup_bh_over_brute']:10.2f}"
    )
print(f"Saved {n_csv.name}")

# --- MPI rank scaling at fixed N ---
MPI_N = 512
mpi_sub = resample_state(state, MPI_N, rng)
mpi_pos, mpi_mass, mpi_eps = mpi_sub.pos, mpi_sub.mass, mpi_sub.eps
mpi_rank_rows = []

if mpi_scaling_available():
    rank_values = [1, 2, 4]
    for n_ranks in rank_values:
        for method in ("brute", "bh"):
            if n_ranks == 1:
                t_force = time_mpi_dispatch_force(mpi_pos, mpi_mass, mpi_eps, method)
            else:
                t_force = bench_mpi_ranks(n_ranks, mpi_pos, mpi_mass, mpi_eps, method)
            mpi_rank_rows.append({
                "n": MPI_N,
                "n_ranks": n_ranks,
                "method": method,
                "ms_per_force": t_force * 1e3,
            })
    mpi_csv = ARTIFACTS / "force_scaling_vs_mpi_ranks.csv"
    with open(mpi_csv, "w") as f:
        f.write("n,n_ranks,method,ms_per_force\n")
        for row in mpi_rank_rows:
            f.write(
                f"{row['n']},{row['n_ranks']},{row['method']},{row['ms_per_force']:.6f}\n"
            )
    print(f"\nMPI time per force step at N={MPI_N} [ms]:")
    print(f"{'ranks':>6} {'method':>6} {'ms':>10}")
    for row in mpi_rank_rows:
        print(f"{row['n_ranks']:6d} {row['method']:6s} {row['ms_per_force']:10.3f}")
    print(f"Saved {mpi_csv.name}")
else:
    print("\nMPI scaling skipped (need mpi4py + mpirun + OpenMPI/MPICH).")

# --- Integrator order comparison ---
integrator_rows = []
bench_state = resample_state(state, 256, rng)
for method in ("brute", "bh"):
    for order in (1, 2):
        t_step = time_integration_step(bench_state, method=method, order=order)
        n_force_calls = order
        integrator_rows.append({
            "method": method,
            "order": order,
            "ms_per_step": t_step * 1e3,
            "ms_per_force": (t_step / n_force_calls) * 1e3,
            "force_calls_per_step": n_force_calls,
        })

int_csv = ARTIFACTS / "integrator_order_timing.csv"
with open(int_csv, "w") as f:
    f.write("method,order,ms_per_step,ms_per_force,force_calls_per_step\n")
    for row in integrator_rows:
        f.write(
            f"{row['method']},{row['order']},{row['ms_per_step']:.6f},"
            f"{row['ms_per_force']:.6f},{row['force_calls_per_step']}\n"
        )

print("\nIntegrator timing at N=256 [ms]:")
print(f"{'method':>6} {'order':>5} {'step':>10} {'per-force':>12}")
for row in integrator_rows:
    print(
        f"{row['method']:>6} {row['order']:5d} {row['ms_per_step']:10.3f} {row['ms_per_force']:12.3f}"
    )
print(f"Saved {int_csv.name}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Panel A: serial scaling vs N
ns = [row["n"] for row in n_scaling_rows]
axes[0].loglog(ns, [row["brute_ms"] for row in n_scaling_rows], "o-", label="brute")
axes[0].loglog(ns, [row["bh_ms"] for row in n_scaling_rows], "s-", label=f"BH (θ={BENCH_THETA})")
axes[0].set_xlabel("N")
axes[0].set_ylabel("ms per force step")
axes[0].set_title("Serial force cost vs N")
axes[0].legend()
axes[0].grid(True, which="both", alpha=0.3)

# Panel B: MPI rank scaling (or placeholder message)
if mpi_rank_rows:
    for method, marker in (("brute", "o-"), ("bh", "s-")):
        subset = [r for r in mpi_rank_rows if r["method"] == method]
        ranks = [r["n_ranks"] for r in subset]
        times = [r["ms_per_force"] for r in subset]
        axes[1].plot(ranks, times, marker, label=method)
    axes[1].set_xlabel("MPI ranks")
    axes[1].set_ylabel("ms per force step")
    axes[1].set_title(f"MPI scaling at N={MPI_N}")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(
        0.5,
        0.5,
        "MPI scaling unavailable\n(install mpi4py + mpirun)",
        ha="center",
        va="center",
        transform=axes[1].transAxes,
    )
    axes[1].set_title("MPI scaling (skipped)")

# Panel C: integrator order — step time and per-force time
labels = [f"{row['method']}\norder {row['order']}" for row in integrator_rows]
x = np.arange(len(labels))
width = 0.35
axes[2].bar(x - width / 2, [row["ms_per_step"] for row in integrator_rows], width, label="per step")
axes[2].bar(x + width / 2, [row["ms_per_force"] for row in integrator_rows], width, label="per force eval")
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels, fontsize=8)
axes[2].set_ylabel("ms")
axes[2].set_title("Leapfrog order 1 vs 2")
axes[2].legend()
axes[2].grid(True, axis="y", alpha=0.3)

fig.tight_layout()
bench_plot_path = ARTIFACTS / "parallelism_scaling.png"
fig.savefig(bench_plot_path, dpi=150)
plt.show()
print(f"Saved {bench_plot_path.name}")

## 5. Stability plot — is the GalactICS IC in equilibrium?

### Why this plot?

This is the **core integrated test**: GalactICS claims the particles are drawn from an equilibrium DF in the halo potential. If we place those particles in ntropy and let them interact via **self-gravity** (with Plummer softening), a good IC should show:

- **Small drift** in ρ(r) over a few dynamical times
- **Small relative energy error** |ΔE/E₀| from the softened N-body Hamiltonian
- No systematic steepening or flattening of the profile

A bad IC (wrong DF, unit mismatch, non-equilibrium sampling) will show rapid profile evolution — particles phase-mix incorrectly, the core contracts or expands, and shell densities change by order unity.

### Energy conservation (Binney & Tremaine)

For a collisionless system the total energy should be conserved by a good symplectic integrator (modulo softening and finite timestep error). Following the standard diagnostic used throughout *Binney & Tremaine* (e.g. their N-body timestep tests), we track

\[
\left|\frac{E(t) - E_0}{E_0}\right|
\]

where E₀ is the initial total energy (kinetic + softened potential, G = 1). `Simulation.run()` records E after every step in `result.energies`.

**Symplectic vs explicit integrators:** leapfrog (order 1 and 2) is *symplectic* — it nearly preserves the Hamiltonian for small Δt. Forward **Euler** and **Runge–Kutta** schemes are explicit but *not symplectic*; they incur systematic energy drift even when the force field is exact. Higher-order RK reduces *local truncation error* but does not make the scheme symplectic: RK4 can still wander off the true energy surface over many steps.

### Run settings

- **Brute force** — exact gravity at this N, isolates IC quality from tree approximation error.
- **25 steps × dt = 0.002** — short deliberately; we are not simulating a Hubble time, only checking that the IC does not explode immediately.
- **Same binning as §2** — direct before/after comparison on the same radial grid.

The printed `max_drift` is the maximum relative change in any shell with enough particles; values ≲ 0.3 are typical for small N with softening (see `test_galacticsics_integration.py`).

In [ ]:
def relative_energy_error(energies: list[float]) -> np.ndarray:
    """|E(t) - E0| / |E0| in the Binney & Tremaine style."""
    e = np.asarray(energies, dtype=float)
    e0 = e[0]
    denom = max(abs(e0), 1e-30)
    return np.abs(e - e0) / denom


STABILITY_DT = 0.002
STABILITY_STEPS = 25

cfg = RunConfig()
cfg.integrator = IntegratorConfig(
    type="leapfrog", dt=STABILITY_DT, n_steps=STABILITY_STEPS, order=2
)
cfg.force = ForceConfig(method="brute")
cfg.parallel = ParallelConfig(enabled=False)
cfg.output.write_final = False
cfg.output.every = 0

result = Simulation(cfg, state=state.copy()).run()

init_prof = bin_spherical_density(
    result.initial_state.pos, result.initial_state.mass, n_bins=20, r_max=r_max
)
final_prof = bin_spherical_density(
    result.final_state.pos, result.final_state.mass, n_bins=20, r_max=r_max
)
max_drift = compare_density_profiles(init_prof, final_prof, min_count=3)

rel_e = relative_energy_error(result.energies)
times = np.arange(len(result.energies)) * STABILITY_DT
e0, e_final = result.energies[0], result.energies[-1]

print(f"E0 = {e0:.6e},  E_final = {e_final:.6e}")
print(f"Final |ΔE/E0| = {rel_e[-1]:.3e}")
print(f"Max |ΔE/E0| over run = {rel_e.max():.3e}")
print(f"Max relative spherical density change: {max_drift:.3f}")

energy_csv = ARTIFACTS / "energy_drift_order2.csv"
with open(energy_csv, "w") as f:
    f.write("step,time,energy,rel_energy_error\n")
    for step, (t, energy, err) in enumerate(zip(times, result.energies, rel_e)):
        f.write(f"{step},{t:.6f},{energy:.12e},{err:.12e}\n")
print(f"Saved {energy_csv.name}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].loglog(init_prof.r_mid, init_prof.rho, "o-", label="t = 0 (GalactICS IC)")
axes[0].loglog(
    final_prof.r_mid,
    final_prof.rho,
    "s-",
    label=f"t = {STABILITY_STEPS}×{STABILITY_DT} (ntropy)",
)
axes[0].set_xlabel("r [kpc]")
axes[0].set_ylabel("ρ [M_unit / kpc³]")
axes[0].set_title("Density stability")
axes[0].legend()

axes[1].semilogy(times, rel_e, "o-", ms=4)
axes[1].set_xlabel("time [GalactICS units]")
axes[1].set_ylabel(r"$|E - E_0| / |E_0|$")
axes[1].set_title("Energy conservation (order-2 leapfrog)")
axes[1].grid(True, which="both", alpha=0.3)

fig.tight_layout()
stability_path = ARTIFACTS / "density_and_energy_stability.png"
fig.savefig(stability_path, dpi=150)
plt.show()
print(f"Saved {stability_path.name}")

### Symplectic vs explicit integrators

The same Δt and force backend (brute) for every scheme:

| Integrator | Symplectic? | Force evals / step |
|------------|-------------|-------------------|
| leapfrog order 2 | yes | 2 |
| leapfrog order 1 | yes | 1 |
| Euler | no | 1 |
| RK2 | no | 2 |
| RK3 | no | 3 |
| RK4 | no | 4 |

Binney & Tremaine use |ΔE/E₀| vs time to judge timestep quality. Symplectic leapfrog curves should stay lowest; Euler and RK should drift systematically even when RK4 has smaller *local* truncation error than Euler.

In [ ]:
from ntropy.integrators.registry import integrator_force_evaluations, is_symplectic

INTEGRATOR_COMPARE_STEPS = 80
INTEGRATOR_SPECS = [
    ("leapfrog", 2, "leapfrog-2 (symplectic)"),
    ("leapfrog", 1, "leapfrog-1 (symplectic)"),
    ("euler", 2, "Euler"),
    ("rk2", 2, "RK2"),
    ("rk3", 2, "RK3"),
    ("rk4", 2, "RK4"),
]

energy_compare_rows = []

fig, ax = plt.subplots(figsize=(8, 4.5))
for itype, order, label in INTEGRATOR_SPECS:
    cfg_cmp = RunConfig()
    cfg_cmp.integrator = IntegratorConfig(
        type=itype,
        order=order,
        dt=STABILITY_DT,
        n_steps=INTEGRATOR_COMPARE_STEPS,
    )
    cfg_cmp.force = ForceConfig(method="brute")
    cfg_cmp.parallel = ParallelConfig(enabled=False)
    cfg_cmp.output.write_final = False
    cfg_cmp.output.every = 0

    run = Simulation(cfg_cmp, state=state.copy()).run()
    rel = relative_energy_error(run.energies)
    t = np.arange(len(run.energies)) * STABILITY_DT
    linestyle = "-" if is_symplectic(itype) else "--"
    ax.semilogy(t, rel, linestyle, ms=3, linewidth=1.5, label=label)

    n_forces = integrator_force_evaluations(itype, order=order)
    for step, (time_val, energy, err) in enumerate(zip(t, run.energies, rel)):
        energy_compare_rows.append({
            "type": itype,
            "order": order,
            "symplectic": is_symplectic(itype),
            "force_evals_per_step": n_forces,
            "step": step,
            "time": time_val,
            "energy": energy,
            "rel_energy_error": err,
        })

    print(
        f"{label:24s}  max |ΔE/E0| = {rel.max():.3e}  "
        f"final = {rel[-1]:.3e}  forces/step = {n_forces}"
    )

ax.set_xlabel("time [GalactICS units]")
ax.set_ylabel(r"$|E - E_0| / |E_0|$")
ax.set_title(f"Symplectic vs explicit integrators (Δt={STABILITY_DT}, brute force)")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
energy_compare_path = ARTIFACTS / "energy_drift_symplectic_vs_explicit.png"
fig.savefig(energy_compare_path, dpi=150)
plt.show()

compare_csv = ARTIFACTS / "energy_drift_by_integrator.csv"
with open(compare_csv, "w") as f:
    f.write(
        "type,order,symplectic,force_evals_per_step,step,time,energy,rel_energy_error\n"
    )
    for row in energy_compare_rows:
        f.write(
            f"{row['type']},{row['order']},{int(row['symplectic'])},"
            f"{row['force_evals_per_step']},{row['step']},{row['time']:.6f},"
            f"{row['energy']:.12e},{row['rel_energy_error']:.12e}\n"
        )
print(f"Saved {energy_compare_path.name}")
print(f"Saved {compare_csv.name}")

## 6. Blog series — where to go next

This notebook is intentionally small and reproducible. Future posts in the series might cover:

| Topic | Package | Notes |
|-------|---------|-------|
| Disk + halo from reference artifacts | `galacticsics` → `ntropy` | `sample_galacticsics_galaxy()` + `tests/generated/reference` |
| Halo-first two-step workflow | `galacticsics` | `examples/halo_first_workflow.py` |
| Exponential disk Σ(R) tests | `galacticsics` gendisk → `ntropy` | `test_galacticsics_integration.py` |
| MPI domain decomposition | `ntropy` | §4 scaling plots; `mpirun -n 4 ntropy-run config.json` |
| Fitting NFW to particles | `galacticsics.fitting` | observational rotation curves |

**Cursor angle:** the rewrite isolates Fortran under `legacy/`, exposes typed Python APIs, replaces checked-in golden files with generated artifacts, and uses **integrated tests** (`ntropy/tests/test_galacticsics_integration.py`) to prove the old IC code still works in the new stack.

> **Note:** `ntropy.ics.*` provides pure-Python analytic samplers for fast unit tests without a legacy build. Production validation should use the **galacticsics** path shown here.

All artifacts from this run live under `notebooks/artifacts/nfw_walkthrough/` and are safe to delete and regenerate.